# Fitting an astrometric orbit, the open way

This tutorial is the **astrometric sibling** of `fit_rv_orbit.ipynb`.
It fits a Gaia-style epoch-astrometry orbit to synthetic along-scan data
in the spirit of the
[emcee line-fitting tutorial](https://emcee.readthedocs.io/en/stable/tutorials/line/):
everything is explicit, editable, and close to the data. We progress
from a Thiele-Innes period scan, to a closed-form least-squares
baseline, to a quick maximum-likelihood point, to a hand-written
probability and a bare `emcee` run.

Gaia measures only the **along-scan** (one-dimensional) projection of
each transit. At a *fixed* non-linear orbital shape `(P, e, tau)` the
along-scan model is **linear** in the four Thiele-Innes amplitudes
`(A, B, F, G)` plus the five astrometric nuisances
`(ra_offset, dec_offset, pmra, pmdec, plx)` -- nine linear amplitudes
in all. We exploit this just like
the RV notebook exploited the linear `(gamma, K cos w, K sin w)`
structure: we sample only the three non-linear shape parameters and
solve the linear amplitudes in closed form *inside* the objective. This
keeps the sampler in three dimensions and sidesteps the high-dimensional
basin-trap that bites the full 14-parameter Thiele-Innes engine fit (the
Campbell basis has 17).

We never touch the production sampler's internal latent
reparametrization. We only *call* audited, public `orblet` functions
for the forward model, the linear solve (whose closed-form marginal
evidence is the likelihood we sample), and the Thiele-Innes -> Campbell
inversion, plus standard `numpy`, `scipy`, `emcee`, and `corner`.

**Units and conventions** (reused from the audited model, not
re-derived):

- `P` orbital period in **days**; internally converted to Keplerian
  years (`P_yr = P / 365.25`) at the model interface.
- `e` eccentricity, dimensionless, `0 <= e < 1`.
- `tau` periastron phase fraction in `[0, 1)`; `tp = tau * P + t_ref`.
- `A, B, F, G` photocenter Thiele-Innes amplitudes in **mas** (they
  already absorb the `-m_comp / M_total * plx` photocenter factor).
- Derived Campbell elements: inclination `i` in `[0, pi]` (**full
  sphere**, the Gaia convention), longitude of ascending node
  `Omega` and argument of periastron `omega` in **radians**, photocenter
  semi-major axis `a_phot` in **mas**.
- Along-scan projection (audited `along_scan_model`):
  `model = d_ra * sin(psi) + d_dec * cos(psi) + 5-parameter astrometry`,
  with `psi` the scan angle (radians, counterclockwise from north),
  `pmra = mu_alpha*` already cos-delta-corrected, and the parallax term
  added as `plx * parallax_factor_al`.

Astrometry alone leaves the companion mass prior-driven (the Thiele-Innes
amplitudes constrain the photocenter track, not the mass split); we
report the directly-measured geometry `(P, e, i, Omega, omega, a_phot)`.

> **Which fit is this?** This is the simple **quick-look / linearized baseline**: a closed-form linear solve at a fixed orbital shape, a `scipy` maximum-likelihood refinement, and an optional bare `emcee` run, all written out in the open. A fully non-linear fit, every parameter sampled, is the next step, and it should start from this quick-look (see the Summary).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.optimize import minimize
import emcee
import corner

# Audited, public building blocks. We use ONLY the forward model, the
# linear solve (whose marginal score IS the likelihood, see section 5)
# and the Thiele-Innes -> Campbell inversion -- NOT the production
# samplers -- so the probability we sample is written out explicitly.
from orblet.simulate.bundles import load_simulated_inputs
from orblet import (
    linear_solve_ti,
    ti_to_kepler,
    scan_ti_frequency,
    ti_design_matrix,
)
from orblet.model import (
    along_scan_model,
    thiele_innes_xy,
    kepler_xy_orbit,
    tp_from_disk_angle,
)
from orblet.constants import MJD_J2010_TCB, DAYS_PER_KEPLER_YEAR

rng = np.random.default_rng(0)

## 1. The data

We load the synthetic demo bundle (the toy orbit, strongly detected in both channels) and
pull out the epoch astrometry. The bundle is fully in-memory and
synthetic -- no files are read, no real targets are touched.

For each transit we have the observation time `obs_time` (days from
J2010.0), the scan angle `scan_angle` (`psi`, radians), the parallax
factor `parallax_factor_al` (dimensionless), the measured along-scan
abscissa `centroid_pos` (mas), and its uncertainty `centroid_pos_err`
(mas). The audited model works in MJD, so we add the J2010 anchor
`MJD_J2010_TCB`. We adopt the bundle's own reference epoch
`t_ref_mjd` as our `EPOCH_REF` (it sets both the periastron-time and
proper-motion zero points); the recovered orbit shape is independent of
this choice because proper motion enters linearly.

In [ ]:
bundle = load_simulated_inputs(seed=0)  # the toy orbit on the demo cadence
ad = bundle.astro_data

t_mjd = np.asarray(ad['obs_time'], dtype=float) + MJD_J2010_TCB   # MJD (TCB)
psi = np.asarray(ad['scan_angle'], dtype=float)                   # rad
pf = np.asarray(ad['parallax_factor_al'], dtype=float)            # dimensionless
d_obs = np.asarray(ad['centroid_pos'], dtype=float)               # mas (along scan)
sigma = np.asarray(ad['centroid_pos_err'], dtype=float)           # mas

EPOCH_REF = float(bundle.truth.t_ref_mjd)  # reference epoch (MJD)

print(f'{t_mjd.size} transits over {t_mjd.max() - t_mjd.min():.0f} days')
print(f'reference epoch EPOCH_REF = {EPOCH_REF:.1f} MJD')

In [ ]:
# The raw along-scan abscissa vs time, with per-epoch +/-1 sigma bars.
# This is the one-dimensional quantity Gaia actually measures; the full
# orbit + parallax + proper motion are all folded into it.
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(t_mjd - EPOCH_REF, d_obs, yerr=sigma, fmt='o', color='k',
            ms=4, capsize=2)
ax.set_xlabel('time - EPOCH_REF (days)')
ax.set_ylabel('along-scan abscissa (mas)')
ax.set_title('Synthetic Gaia epoch astrometry (along-scan)')
plt.show()

## 2. Period search (Thiele-Innes frequency scan)

The astrometric analogue of the periodogram is the audited
`scan_ti_frequency`: a coarse 3-D `(frequency, e, tau)` linear-TI scan.
At every grid node it builds the along-scan design matrix and runs the
generalised-least-squares solve, ranking nodes by the flat-prior
marginal-likelihood score `logL_marginal`. Unlike a plain periodogram it
is **parallax- and proper-motion-aware** (those columns are in the
design matrix), which matters because the Gaia scanning law makes the
**1-year / 6-month parallax aliases** intrinsically degenerate with the
parallax column.

The scan **detects and flags** peaks in the 1-year alias region rather
than solving them away -- an honest warning, not a fix. For this
synthetic orbit the true period (~185 d) happens to sit near the
**6-month** parallax alias, so the global-best peak is itself flagged:
the flag tells you to be careful, not that the peak is wrong. We plot the
ranked peaks and mark the alias regions.

In [ ]:
ecc_grid = [0.0, 0.2, 0.4, 0.6]
tau_grid = np.linspace(0.0, 1.0, 8, endpoint=False)

peaks = scan_ti_frequency(
    t_mjd, psi, pf, d_obs, sigma,
    f_min_per_day=1.0 / 500.0,   # search 50..500 day periods
    f_max_per_day=1.0 / 50.0,
    oversample=4.0,
    ecc_grid=ecc_grid,
    tau_grid=tau_grid,
    epoch_ref_mjd=EPOCH_REF,
    top_k=6,
)

print('rank   P (d)     e    tau    logL_marginal   alias-flagged')
for p in peaks:
    print(f'      {p.P_days:8.2f}  {p.ecc:.2f}  {p.tau:.2f}   '
          f'{p.logL_marginal:12.1f}   {p.flagged}')

# Period seed = the TOP-ranked peak by logL_marginal. Note that for this
# synthetic orbit the true period (~185 d) sits close to the 6-month
# parallax alias, so the global-best peak is itself alias-flagged. The
# flag is a REVIEW warning, not a disqualifier: it is still the best
# linear fit, and the quick ML refine below confirms it. (Compare: the
# best NON-flagged peak near ~224 d is a much worse fit.)
P_scan = float(peaks[0].P_days)
print(f'\ntop-ranked peak: P = {peaks[0].P_days:.2f} d '
      f'(alias-flagged = {peaks[0].flagged})')

In [ ]:
# logL(period) for the ranked peaks, with the 1-year / 6-month parallax
# alias regions marked. The scan flags peaks that fall in these regions.
P_one_year = 365.25
P_six_month = 365.25 / 2.0

fig, ax = plt.subplots(figsize=(8, 4))
P_peaks = np.array([p.P_days for p in peaks])
L_peaks = np.array([p.logL_marginal for p in peaks])
flagged = np.array([p.flagged for p in peaks])
ax.scatter(P_peaks[~flagged], L_peaks[~flagged], color='k', zorder=5,
           label='ranked peaks')
if flagged.any():
    ax.scatter(P_peaks[flagged], L_peaks[flagged], color='C3', marker='x',
               s=60, zorder=6, label='alias-flagged')
for P_alias, lab in [(P_one_year, '1 yr'), (P_six_month, '6 mo')]:
    ax.axvline(P_alias, color='C3', ls='--', alpha=0.6)
    ax.text(P_alias, ax.get_ylim()[1], f' {lab} alias', color='C3',
            va='top', ha='left', fontsize=8)
ax.axvline(P_scan, color='C0', ls=':', label=f'P seed = {P_scan:.1f} d')
ax.set_xlabel('period (days)')
ax.set_ylabel('logL_marginal (ranking score)')
ax.legend(fontsize=8)
plt.show()

## 3. Closed-form least-squares baseline

At a **fixed** non-linear shape `(P, e, tau)`, the along-scan model is
*linear* in the nine amplitudes
`beta = [A, B, F, G, ra_offset, dec_offset, pmra, pmdec, plx]`. The
audited `_ti_design_matrix` builds the 9-column design (its columns are
the partial derivatives of `along_scan_model`), and `linear_solve_ti`
returns `beta` from a single generalised-least-squares solve -- no
sampling needed.

We then convert the four Thiele-Innes amplitudes to Campbell elements
`(i, Omega, omega, a_phot)` with the audited `ti_to_kepler`, wrapping the
scalars as length-1 arrays exactly as the production `_ti9h_skyplane`
helper does. This is a **point estimate**, and two different things are
often conflated here:

- the **sign gauge** `(A,B,F,G) -> -(A,B,F,G)`, which maps
  `(omega, Omega) -> (omega + pi, Omega + pi)` and is the *same* orbit.
  It is bookkeeping, folded away at chain export by keeping the
  representative with `Omega` in `[0, pi)`;
- the true **photocentre mirror** `(i, omega, Omega) <-> (pi - i, pi - omega, pi - Omega)`,
  a genuinely different geometry that predicts the *same* along-scan
  data. Astrometry alone cannot break it; radial velocities can.

In [ ]:
def solve_linear(P_days, e, tau):
    """Closed-form 9 amplitudes at fixed shape via the audited GLS core."""
    X = ti_design_matrix(t_mjd, psi, pf, f_per_day=1.0 / P_days,
                          ecc=e, tau=tau, epoch_ref_mjd=EPOCH_REF)
    sol = linear_solve_ti(d_obs, sigma, X)
    return sol, X


def campbell_from_beta(beta):
    """Thiele-Innes (A,B,F,G)+plx -> Campbell (i, Omega, omega, a_phot).

    Wrap the scalars as length-1 arrays and call the audited chain-array
    inverter ti_to_kepler (the same Thiele-Innes -> Campbell recipe the
    production astrometric engine uses).
    """
    kep = ti_to_kepler(
        {'A_mas': beta[0:1], 'B_mas': beta[1:2], 'F_mas': beta[2:3],
         'G_mas': beta[3:4], 'plx_mas': beta[8:9]},
        plx_key='plx_mas',
    )
    return {
        'i_deg': float(np.rad2deg(kep['inc_rad'][0])),
        'Omega_deg': float(np.rad2deg(kep['Omega_rad'][0])),
        'omega_deg': float(np.rad2deg(kep['omega_rad'][0])),
        'a_phot_mas': float(kep['a_phot_mas'][0]),
    }


# Scan tau at (P_scan, e=0.4) for the best-fitting linear amplitudes.
e_guess = 0.4
tau_grid_ls = np.linspace(0.0, 1.0, 200, endpoint=False)
chi2_grid = np.array([solve_linear(P_scan, e_guess, tau)[0].chi2
                      for tau in tau_grid_ls])
tau_ls = float(tau_grid_ls[np.argmin(chi2_grid)])
sol_ls, _ = solve_linear(P_scan, e_guess, tau_ls)
camp_ls = campbell_from_beta(sol_ls.beta)
print(f'LS baseline at P={P_scan:.2f} d, e={e_guess:.2f}, tau={tau_ls:.3f}:')
print(f'  i = {camp_ls["i_deg"]:.1f} deg  (point estimate; the i <-> 180-i photocentre')
print('      mirror is a real degeneracy of along-scan-only data)')
print(f'  Omega = {camp_ls["Omega_deg"]:.1f} deg')
print(f'  omega = {camp_ls["omega_deg"]:.1f} deg')
print(f'  a_phot = {camp_ls["a_phot_mas"]:.3f} mas')

We trace the quick-fit photocenter ellipse on the sky plane, mirroring
the audited `_ti9h_skyplane` recipe: the orbit is drawn with the engine
projection `d_ra = B*x + G*y`, `d_dec = A*x + F*y`; the data points are
the centroid minus the fitted 5-parameter (offset + proper-motion +
parallax) model. Each per-epoch error bar is the **1-D along-scan +/-1
sigma**, oriented along the scan unit vector `(sin psi, cos psi)` with
half-length `centroid_pos_err`. The across-scan coordinate of each
plotted point is **model-imputed** -- Gaia measures along-scan only -- so
the visual scatter is along-scan-direction-only.

In [ ]:
def plot_skyplane(P_days, e, tau, beta, ax=None, title=None):
    """Photocenter sky-plane orbit + AL error bars from the fitted T-I amplitudes."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    P_yr = P_days / DAYS_PER_KEPLER_YEAR
    A_ti, B_ti, F_ti, G_ti = beta[0], beta[1], beta[2], beta[3]
    ra_off, dec_off, pmra_s, pmdec_s, plx_s = beta[4:9]

    disk_x, disk_y = np.cos(2 * np.pi * tau), np.sin(2 * np.pi * tau)
    tp = tp_from_disk_angle(disk_x, disk_y, period_days=P_days,
                             epoch_ref_mjd=EPOCH_REF)

    # Smooth orbit ellipse via the engine projection d_ra=B*x+G*y, d_dec=A*x+F*y.
    t_curve = np.linspace(0.0, P_days, 400) + tp
    x_c, y_c = kepler_xy_orbit(t_curve, period_yr=P_yr, ecc=e, tp_mjd=tp)
    dRA_curve = B_ti * x_c + G_ti * y_c
    dDec_curve = A_ti * x_c + F_ti * y_c

    # Orbit-only data: subtract the fitted 5-parameter astrometry columns.
    X = ti_design_matrix(t_mjd, psi, pf, f_per_day=1.0 / P_days, ecc=e,
                          tau=tau, epoch_ref_mjd=EPOCH_REF)
    model_5p = (ra_off * X[:, 4] + dec_off * X[:, 5] + pmra_s * X[:, 6]
                + pmdec_s * X[:, 7] + plx_s * X[:, 8])
    w_orbit = d_obs - model_5p  # along-scan orbit-only residual [mas]

    x_o, y_o = kepler_xy_orbit(t_mjd, period_yr=P_yr, ecc=e, tp_mjd=tp)
    dRA_model = B_ti * x_o + G_ti * y_o
    dDec_model = A_ti * x_o + F_ti * y_o
    sin_psi, cos_psi = np.sin(psi), np.cos(psi)
    # Across-scan (AC) unit vector = scan vector rotated +90 deg: (cos, -sin).
    # AC coordinate is model-imputed (Gaia measures along-scan only).
    AC_model = dRA_model * cos_psi - dDec_model * sin_psi
    data_dRA = w_orbit * sin_psi + AC_model * cos_psi
    data_dDec = w_orbit * cos_psi - AC_model * sin_psi

    # Per-epoch AL +/-1 sigma bars: (sin psi, cos psi) is the AL (constrained)
    # direction; half-length = centroid_pos_err.
    seg0 = np.column_stack((data_dRA - sigma * sin_psi,
                            data_dDec - sigma * cos_psi))
    seg1 = np.column_stack((data_dRA + sigma * sin_psi,
                            data_dDec + sigma * cos_psi))
    segs = np.stack((seg0, seg1), axis=1)  # (N, 2 ends, 2 xy)
    ax.add_collection(LineCollection(segs, linewidths=0.8, alpha=0.5,
                                     zorder=3, label='AL +/-1 sigma'))

    ax.plot(dRA_curve, dDec_curve, '-', lw=1.8, color='C0', label='TI orbit')
    ax.scatter(data_dRA, data_dDec, s=14, marker='o', facecolor='white',
               edgecolor='black', lw=0.9, zorder=5, label='AL transits')
    ax.scatter(0.0, 0.0, marker='+', color='k', s=70, zorder=4,
               label='barycenter')
    ax.set_xlabel(r'$\Delta\alpha^{*}$ [mas]')
    ax.set_ylabel(r'$\Delta\delta$ [mas]')
    ax.set_aspect('equal')
    ax.invert_xaxis()
    ax.legend(loc='best', fontsize=8)
    ax.grid(alpha=0.3)
    if title:
        ax.set_title(title)
    return ax


fig, ax = plt.subplots(figsize=(6, 6))
plot_skyplane(P_scan, e_guess, tau_ls, sol_ls.beta, ax=ax,
              title='LS baseline photocenter orbit (across-scan is model-imputed)')
plt.show()

## 4. Quick maximum-likelihood point ('quick and dirty fit')

Now optimise the three *non-linear* shape parameters `(P, e, tau)` with
`scipy.optimize.minimize`, solving the nine linear amplitudes in closed
form inside the objective (the audited `linear_solve_ti` marginal score).
This mirrors the RV notebook: three dimensions, robust, and it avoids the
high-dimensional basin-trap. We try a few starts.

In [ ]:
def neg_marginal_logL(shape):
    P, e, tau = shape
    if not (50.0 < P < 500.0 and 0.0 <= e < 0.9 and 0.0 <= tau < 1.0):
        return 1e12
    try:
        sol, _ = solve_linear(P, e, tau)
    except Exception:
        return 1e12
    return -sol.logL_marginal


best = None
for e0 in (0.1, 0.4, 0.6):
    for tau0 in np.linspace(0.05, 0.95, 5):
        res = minimize(neg_marginal_logL, [P_scan, e0, tau0],
                       method='Nelder-Mead')
        if best is None or res.fun < best.fun:
            best = res

P_ml, e_ml, tau_ml = best.x
sol_ml, _ = solve_linear(P_ml, e_ml, tau_ml)
camp_ml = campbell_from_beta(sol_ml.beta)
print('quick ML point (shape):')
print(f'  P     = {P_ml:.2f} d')
print(f'  e     = {e_ml:.3f}')
print(f'  tau   = {tau_ml:.3f}')
print('derived Campbell geometry:')
print(f'  i      = {camp_ml["i_deg"]:.1f} deg')
print(f'  Omega  = {camp_ml["Omega_deg"]:.1f} deg')
print(f'  omega  = {camp_ml["omega_deg"]:.1f} deg')
print(f'  a_phot = {camp_ml["a_phot_mas"]:.3f} mas')

theta_ml = np.array([P_ml, e_ml, tau_ml])

## 5. An explicit, editable probability

Following the emcee tutorial, we write the three functions out by hand.

The likelihood is the **marginal evidence** at the shape: at fixed
`(P, e, tau)` the along-scan model is linear in the nine amplitudes, so
`linear_solve_ti` integrates them out in closed form and returns
`sol.logL_marginal` (a Gaussian evidence that keeps the `log|M|` Occam
term). That is **exactly the score step 4 optimised**, so the sampler and
the quick ML point now target the same function.

This matters: evaluating the Gaussian at the fitted amplitudes instead
would be the *profile* likelihood -- the amplitudes' uncertainty dropped
rather than integrated -- which is a different (over-confident) target.
`fit_rv_orbit.ipynb` §5 makes the same choice for the RV channel.

`model_along_scan_from_shape` below still rebuilds the model from the
audited atoms (`thiele_innes_xy` + `along_scan_model`); we use it for the
residual plots in step 7, not inside the likelihood. The prior is a
simple editable box, and the jitter is held FIXED at zero throughout (the
quick-look's standing assumption; `fit_astrometric_orbit_nonlinear.ipynb`
frees it).

In [ ]:
# Box prior bounds on the non-linear shape theta = (P, e, tau).
P_LO, P_HI = 150.0, 250.0  # days

def model_along_scan_from_shape(P_days, e, tau):
    """Along-scan model (mas) at a shape, solving the 9 amplitudes inside."""
    sol, _ = solve_linear(P_days, e, tau)
    beta = sol.beta
    P_yr = P_days / DAYS_PER_KEPLER_YEAR
    disk_x, disk_y = np.cos(2 * np.pi * tau), np.sin(2 * np.pi * tau)
    tp = tp_from_disk_angle(disk_x, disk_y, period_days=P_days,
                             epoch_ref_mjd=EPOCH_REF)
    d_ra, d_dec = thiele_innes_xy(t_mjd, period_yr=P_yr, ecc=e,
                                  A_mas=beta[0], B_mas=beta[1],
                                  F_mas=beta[2], G_mas=beta[3], tp_mjd=tp)
    model = along_scan_model(
        d_ra=d_ra, d_dec=d_dec, psi=psi, t_mjd=t_mjd, epoch_ref_mjd=EPOCH_REF,
        ra_offset_mas=beta[4], dec_offset_mas=beta[5],
        pmra_masyr=beta[6], pmdec_masyr=beta[7], plx_mas=beta[8],
        parallax_factor_al=pf,
    )
    return model, beta


def log_prior(theta):
    P, e, tau = theta
    if P_LO < P < P_HI and 0.0 <= e < 0.9 and 0.0 <= tau < 1.0:
        return 0.0
    return -np.inf


def log_likelihood(theta):
    """MARGINAL log-likelihood at the shape: amplitudes integrated out.

    The same `sol.logL_marginal` score step 4 optimised. Scoring the
    Gaussian at the fitted amplitudes would be the PROFILE likelihood --
    over-confident, and a different target from the ML point above.
    """
    P, e, tau = theta
    try:
        sol, _ = solve_linear(P, e, tau)
    except Exception:
        return -np.inf
    return float(sol.logL_marginal)


def log_probability(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)


print('log_probability at the quick ML point:', log_probability(theta_ml))

## 6. Sample the posterior with emcee

A bare `emcee.EnsembleSampler` seeded in a tiny Gaussian ball around the
quick ML point. We run a modest chain, discard a burn-in, and look at the
traces and a corner plot of the three sampled shape parameters.

In [ ]:
n_walkers = 24
n_dim = 3
n_steps = 1200

p0 = theta_ml + 1e-3 * rng.standard_normal((n_walkers, n_dim))
p0[:, 1] = np.clip(p0[:, 1], 0.0, 0.89)   # keep e in support
p0[:, 2] = np.clip(p0[:, 2], 0.0, 0.999)  # keep tau in [0, 1)

sampler = emcee.EnsembleSampler(n_walkers, n_dim, log_probability)
sampler.run_mcmc(p0, n_steps, progress=True)
print('mean acceptance fraction:', np.mean(sampler.acceptance_fraction))

In [ ]:
labels = ['P (d)', 'e', 'tau']
fig, axes = plt.subplots(n_dim, figsize=(8, 5), sharex=True)
chain = sampler.get_chain()
for i in range(n_dim):
    axes[i].plot(chain[:, :, i], color='k', alpha=0.3, lw=0.5)
    axes[i].set_ylabel(labels[i])
axes[-1].set_xlabel('step')
plt.show()

In [ ]:
burnin = 400
thin = 4
flat = sampler.get_chain(discard=burnin, thin=thin, flat=True)
fig = corner.corner(flat, labels=labels, show_titles=True,
                    title_fmt='.3f', quantiles=[0.16, 0.5, 0.84])
plt.show()

## 7. Look at the data: sky-plane orbit + residuals

We push every posterior shape sample back through the closed-form linear
solve and **draw** the amplitudes from their conditional Gaussian
`N(sol.beta, sol.cov)` (a point estimate would under-disperse the derived
geometry) to get the implied Campbell elements `(i, Omega, omega, a_phot)`,
then report 16/50/84 percentiles for `(P, e, i, Omega, omega, a_phot)`
against the injected truth. Finally we overlay the posterior-median orbit
on the sky plane (with the same along-scan error bars as step 3) and plot
the along-scan residuals against both time and scan angle.

In [ ]:
# Per-sample amplitudes -> Campbell geometry. At a fixed shape the
# conditional posterior of the nine amplitudes is the Gaussian
# N(sol.beta, sol.cov), so we DRAW from it rather than taking the point
# estimate sol.beta: drawing carries the amplitudes' proper conditional
# uncertainty into (i, Omega, omega, a_phot), which a point estimate
# would make artificially sharp (under-dispersed). Same recipe as
# fit_rv_orbit.ipynb §5.
betas = np.array([
    rng.multivariate_normal(sol.beta, sol.cov)
    for sol in (solve_linear(P, e, tau)[0] for (P, e, tau) in flat)
])
kep = ti_to_kepler(
    {'A_mas': betas[:, 0], 'B_mas': betas[:, 1], 'F_mas': betas[:, 2],
     'G_mas': betas[:, 3], 'plx_mas': betas[:, 8]},
    plx_key='plx_mas',
)
i_deg = np.rad2deg(kep['inc_rad'])
Omega_deg = np.rad2deg(kep['Omega_rad'])
omega_deg = np.rad2deg(kep['omega_rad'])
a_phot_mas = kep['a_phot_mas']

truth = bundle.truth
report = [
    ('P (d)',       flat[:, 0],  truth.P_days),
    ('e',           flat[:, 1],  truth.e),
    ('i (deg)',     i_deg,       np.rad2deg(truth.i_rad)),
    ('Omega (deg)', Omega_deg,   np.rad2deg(truth.Omega_rad)),
    ('omega (deg)', omega_deg,   np.rad2deg(truth.omega_rad)),
    ('a_phot (mas)', a_phot_mas, truth.a_phot_mas),
]
print('parameter         16%       50%       84%      truth')
pct = {}
for name, arr, tv in report:
    q = np.percentile(arr, [16, 50, 84])
    pct[name] = q
    print(f'{name:14s} {q[0]:9.3f} {q[1]:9.3f} {q[2]:9.3f}  {tv:9.3f}')

In [ ]:
# Median-shape sky-plane orbit (re-using the step-3 helper).
P_med = float(pct['P (d)'][1])
# Pick the posterior sample closest to the median period; trace its orbit.
i_med = int(np.argmin(np.abs(flat[:, 0] - P_med)))
P_s, e_s, tau_s = flat[i_med]
sol_s, _ = solve_linear(P_s, e_s, tau_s)

fig, ax = plt.subplots(figsize=(6, 6))
plot_skyplane(P_s, e_s, tau_s, sol_s.beta, ax=ax,
              title='Posterior-median photocenter orbit')
fig.text(0.5, 0.005,
         'across-scan coordinate is model-imputed (along-scan-only data); '
         'error bars are 1-D along-scan +/-1 sigma',
         ha='center', va='bottom', fontsize=7, color='0.4')
plt.show()

In [ ]:
# Along-scan residuals at the median shape, vs time and vs scan angle.
model_med, _ = model_along_scan_from_shape(P_s, e_s, tau_s)
resid = d_obs - model_med  # mas

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.errorbar(t_mjd - EPOCH_REF, resid, yerr=sigma, fmt='o', color='k',
             ms=4, capsize=2)
ax1.axhline(0.0, color='C3', lw=1)
ax1.set_xlabel('time - EPOCH_REF (days)')
ax1.set_ylabel('along-scan residual (mas)')
ax1.set_title('Residual vs time')

ax2.errorbar(np.rad2deg(psi), resid, yerr=sigma, fmt='o', color='k',
             ms=4, capsize=2)
ax2.axhline(0.0, color='C3', lw=1)
ax2.set_xlabel('scan angle psi (deg)')
ax2.set_ylabel('along-scan residual (mas)')
ax2.set_title('Residual vs scan angle')
plt.tight_layout()
plt.show()

print(f'residual RMS: {np.sqrt(np.mean(resid ** 2)):.4f} mas')

## Summary

We recovered the injected astrometric orbit (typical values from a run of
this notebook: `P ~ 185 d`, `e ~ 0.45`, `i ~ 127 deg`, `a_phot ~ 2.7 mas`
-- read the table §7 prints, not these) starting from one-dimensional Gaia
along-scan abscissae, using only standard tools plus audited `orblet`
forward-model, linear-solve, Thiele-Innes -> Campbell, and Gaussian
likelihood functions. Every step -- the TI period scan, the closed-form
baseline, the quick ML, the explicit prior/likelihood/probability, and
the emcee run -- is laid out in the open so you can edit priors, bounds,
and the model freely.

Two honesty caveats carried from the audited code:

- The **inclination** from a single solve is a point estimate carrying
  the photocentre mirror `(i, omega, Omega) <-> (180-i, 180-omega, 180-Omega)`
  -- a genuinely different geometry with the same along-scan prediction
  (distinct from the `(A,B,F,G) -> -(A,B,F,G)` sign gauge, which is the
  same orbit and is folded away by `Omega` in `[0, 180)`). The posterior
  here is informative because the data prefer one branch, but in general
  both mirror modes can appear.
- Gaia measures **only along-scan**: the sky-plane scatter is
  along-scan-direction-only and the across-scan coordinate of each
  plotted point is model-imputed.

Astrometry alone constrains the **geometry**, not the companion mass; the
mass split stays prior-driven. For a fully non-linear fit, SEED it from
this quick-look (`compose_ti_seed(peaks[0], epoch_ref_mjd=EPOCH_REF)`):
the full fit refines, it does not search, and started cold it can miss
the basin this scan found. The seed only sets where the chains start.
§9 of `fit_joint_orbit.ipynb` shows the all-parameter rung for the joint
case.